In [3]:
# ============================================================
# LABEL ASSISTED FUZZY OBIA MOUND DETECTOR
# Probability from original calibrated fuzzy OBIA
# Candidate extraction with mask label object classifier
# For TAG style SMS mound morphology detection from bathymetry
# ============================================================
#
# Important:
# This script detects mound morphology only.
# It does not prove that a mound is SMS.
#
# The fuzzy probability part is kept the same as the earlier
# calibrated OBIA version.
#
# The extraction part now uses mask_label.gpkg to learn which
# probability objects are real mounds and which are false positives.
#
# Expected label file:
# mask_label.gpkg
# layer: traintest
# column: label
# label = 1 means mound
# label = 0 means false positive or non mound
#
# If no label 0 polygons are available, the script creates pseudo
# negative objects from very elongated or weakly mound shaped objects.
# ============================================================


# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize, shapes, geometry_mask

from scipy.ndimage import (
    uniform_filter,
    gaussian_filter,
    binary_closing,
    binary_opening,
    binary_fill_holes,
    binary_dilation,
    label as connected_label,
    find_objects,
    distance_transform_edt,
    maximum_filter,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import joblib

try:
    import geopandas as gpd
    from shapely.geometry import shape
    GEOPANDAS_AVAILABLE = True
except Exception:
    GEOPANDAS_AVAILABLE = False

try:
    from skimage.segmentation import watershed
    from skimage.morphology import h_maxima
    SKIMAGE_AVAILABLE = True
except Exception:
    watershed = None
    h_maxima = None
    SKIMAGE_AVAILABLE = False

warnings.filterwarnings("ignore")


# ============================================================
# 2. Paths
# ============================================================

data_dir = Path(r"")

bathymetry_path = data_dir / "bathymetry_2m_epsg32623_reproject.tif"

label_gpkg = data_dir / "prob_label.gpkg"
label_layer = "label_prob"
label_column = "label"
positive_label_values = [1]

roughness_path = data_dir / "roughness_reproject.tif"
slope_path = data_dir / "slope_map_2m_tag.tif"

output_dir = data_dir / "obia_label_assisted_mound_outputs_new_try_inshallah_LAST"
output_dir.mkdir(parents=True, exist_ok=True)

feature_dir = output_dir / "derived_features"
membership_dir = output_dir / "memberships"
feature_dir.mkdir(exist_ok=True)
membership_dir.mkdir(exist_ok=True)


# ============================================================
# 3. Original probability settings
# ============================================================

bathymetry_sign_mode = "negative_depths"

local_height_windows_m = [80, 120, 180, 240, 300, 400]

roughness_window_m = 20

edge_buffer_m = 25

use_calibration_mask = True
calibration_ignore_roughest_percentile = 85

calibration_also_ignore_extreme_local_height = True
calibration_ignore_local_height_percentile = 97

smooth_probability_sigma_pixels = 2.0
smooth_broad_support_sigma_pixels = 1.8


# ============================================================
# 4. New label assisted extraction settings
# ============================================================

# This mask should be permissive, but not so permissive that nearby mounds become one blob.
candidate_probability_threshold = 0.26
candidate_broad_threshold = 0.18

# Selective watershed splitting is safer than splitting every connected object.
# It only splits large, long, or complex blobs. Small compact objects stay intact.
use_peak_splitting = True
use_selective_peak_splitting = True

# Smaller distance helps close small mounds get separate seed points.
# The prominence filter below prevents one rough mound from getting too many seeds.
minimum_peak_distance_m = 35
seed_peak_probability_threshold = 0.38
seed_peak_broad_threshold = 0.18
seed_peak_prominence = 0.04
split_surface_sigma_pixels = 1.2

# Objects smaller and rounder than these limits are kept as one object.
split_only_area_above_m2 = 12000
split_only_major_axis_above_m = 90
split_only_elongation_above = 2.2

# Maximum growth radius keeps watershed regions from swallowing broad background.
maximum_growth_radius_m = 95

# Keep cleanup gentle. Large closing bridges nearby mounds.
closing_radius_m = 2
opening_radius_m = 2
fill_holes_before_splitting = False

# Object size limits before classification.
minimum_candidate_area_m2 = 500
maximum_candidate_area_m2 = 250000

# Final classifier threshold.
# Lower gives more candidates.
# Higher gives fewer candidates.
final_object_probability_threshold = 0.45

# Training overlap criteria.
min_positive_overlap_area_m2 = 40
min_positive_overlap_fraction = 0.005

min_negative_overlap_area_m2 = 100
min_negative_overlap_fraction = 0.05

# Keep known positive training objects in final output.
force_keep_training_positives = True

# Create false positive examples automatically when label 0 is missing or too few.
create_pseudo_negatives = True
minimum_training_objects_per_class = 3
maximum_pseudo_negative_objects = 250

random_seed = 42

polygon_simplify_tolerance_m = 1.0

run_known_mound_diagnostics = True

object_model_path = output_dir / "label_assisted_object_classifier.joblib"


# ============================================================
# 5. Original fuzzy feature settings
# ============================================================

feature_settings = [
    {
        "name": "local_height_80m",
        "kind": "local_height",
        "window_m": 80,
        "membership": "high",
        "percentiles": (55, 97),
        "weight": 0.03,
        "group": "support",
    },
    {
        "name": "local_height_120m",
        "kind": "local_height",
        "window_m": 120,
        "membership": "high",
        "percentiles": (35, 95),
        "weight": 0.16,
        "group": "broad",
    },
    {
        "name": "local_height_180m",
        "kind": "local_height",
        "window_m": 180,
        "membership": "high",
        "percentiles": (35, 95),
        "weight": 0.23,
        "group": "broad",
    },
    {
        "name": "local_height_240m",
        "kind": "local_height",
        "window_m": 240,
        "membership": "high",
        "percentiles": (35, 95),
        "weight": 0.22,
        "group": "broad",
    },
    {
        "name": "local_height_300m",
        "kind": "local_height",
        "window_m": 300,
        "membership": "high",
        "percentiles": (35, 95),
        "weight": 0.17,
        "group": "broad",
    },
    {
        "name": "local_height_400m",
        "kind": "local_height",
        "window_m": 400,
        "membership": "high",
        "percentiles": (35, 95),
        "weight": 0.08,
        "group": "broad",
    },
    {
        "name": "roughness",
        "kind": "roughness",
        "membership": "high",
        "percentiles": (72, 97),
        "weight": 0.01,
        "group": "support",
    },
]


# ============================================================
# 6. Helper functions
# ============================================================

def now():
    return time.strftime("%H:%M:%S")


def disk_structure(radius_pixels):
    radius_pixels = int(max(1, round(radius_pixels)))
    y, x = np.ogrid[-radius_pixels:radius_pixels + 1, -radius_pixels:radius_pixels + 1]
    return (x * x + y * y) <= radius_pixels * radius_pixels


def clean_output_profile(profile):
    out_profile = profile.copy()
    for key in ["blockxsize", "blockysize", "BLOCKXSIZE", "BLOCKYSIZE"]:
        out_profile.pop(key, None)
    out_profile.update(
        driver="GTiff",
        tiled=True,
        blockxsize=256,
        blockysize=256,
        compress="deflate",
        BIGTIFF="IF_SAFER",
    )
    return out_profile


def write_float_raster(path, array, profile, nodata=-9999.0):
    out_profile = clean_output_profile(profile)
    out_profile.update(dtype="float32", count=1, nodata=nodata, predictor=3)
    out = np.where(np.isfinite(array), array, nodata).astype("float32")
    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(out, 1)


def write_uint8_raster(path, array, profile):
    out_profile = clean_output_profile(profile)
    out_profile.update(dtype="uint8", count=1, nodata=0)
    out = np.where(array > 0, 1, 0).astype("uint8")
    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(out, 1)


def write_int32_raster(path, array, profile):
    out_profile = clean_output_profile(profile)
    out_profile.update(dtype="int32", count=1, nodata=0)
    out = np.where(np.isfinite(array), array, 0).astype("int32")
    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(out, 1)


def same_grid(src, master_profile):
    return (
        src.crs == master_profile["crs"]
        and src.width == master_profile["width"]
        and src.height == master_profile["height"]
        and src.transform.almost_equals(master_profile["transform"])
    )


def read_and_align_raster(raster_path, master_profile, resampling=Resampling.bilinear):
    with rasterio.open(raster_path) as src:
        source = src.read(1).astype("float32")
        if src.nodata is not None:
            source[source == src.nodata] = np.nan
        source[~np.isfinite(source)] = np.nan

        destination = np.full(
            (master_profile["height"], master_profile["width"]),
            np.nan,
            dtype="float32",
        )

        if same_grid(src, master_profile):
            destination = source
        else:
            reproject(
                source=source,
                destination=destination,
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=np.nan,
                dst_transform=master_profile["transform"],
                dst_crs=master_profile["crs"],
                dst_nodata=np.nan,
                resampling=resampling,
            )

    destination[~np.isfinite(destination)] = np.nan
    return destination.astype("float32")


def mean_filter_nan(array, size_pixels):
    size_pixels = int(max(3, round(size_pixels)))
    if size_pixels % 2 == 0:
        size_pixels += 1

    valid = np.isfinite(array)
    data = np.where(valid, array, 0.0).astype("float32")
    weights = valid.astype("float32")

    summed = uniform_filter(data, size=size_pixels, mode="nearest")
    weight_sum = uniform_filter(weights, size=size_pixels, mode="nearest")

    out = np.full(array.shape, np.nan, dtype="float32")
    good = weight_sum > 0.001
    out[good] = summed[good] / weight_sum[good]
    return out


def std_filter_nan(array, size_pixels):
    mean = mean_filter_nan(array, size_pixels)
    mean_sq = mean_filter_nan(array * array, size_pixels)
    variance = mean_sq - mean * mean
    variance = np.where(variance > 0, variance, 0)
    return np.sqrt(variance).astype("float32")


def gaussian_filter_nan(array, sigma_pixels):
    if sigma_pixels <= 0:
        return array.astype("float32")

    valid = np.isfinite(array)
    data = np.where(valid, array, 0.0).astype("float32")
    weights = valid.astype("float32")

    smooth_data = gaussian_filter(data, sigma=sigma_pixels, mode="nearest")
    smooth_weights = gaussian_filter(weights, sigma=sigma_pixels, mode="nearest")

    out = np.full(array.shape, np.nan, dtype="float32")
    good = smooth_weights > 0.001
    out[good] = smooth_data[good] / smooth_weights[good]
    return out


def percentile_membership_high(array, valid_mask, p_low, p_high):
    values = array[valid_mask & np.isfinite(array)]
    out = np.zeros(array.shape, dtype="float32")

    if values.size < 100:
        out[:] = np.nan
        return out, np.nan, np.nan

    low, high = np.nanpercentile(values, [p_low, p_high])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        out[:] = np.nan
        return out, low, high

    out = (array - low) / (high - low)
    out = np.clip(out, 0, 1).astype("float32")
    out[~np.isfinite(array)] = np.nan
    return out, float(low), float(high)


def percentile_membership_low(array, valid_mask, p_low, p_high):
    high_membership, low, high = percentile_membership_high(array, valid_mask, p_low, p_high)
    out = 1.0 - high_membership
    out[~np.isfinite(high_membership)] = np.nan
    return out.astype("float32"), low, high


def trapezoid_membership(array, a, b, c, d):
    out = np.zeros(array.shape, dtype="float32")
    finite = np.isfinite(array)

    rising = finite & (array > a) & (array < b)
    plateau = finite & (array >= b) & (array <= c)
    falling = finite & (array > c) & (array < d)

    if b > a:
        out[rising] = (array[rising] - a) / (b - a)
    out[plateau] = 1.0
    if d > c:
        out[falling] = (d - array[falling]) / (d - c)

    out = np.clip(out, 0, 1).astype("float32")
    out[~finite] = np.nan
    return out


def safe_mean(values):
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    return float(np.nanmean(values))


def safe_max(values):
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    return float(np.nanmax(values))


def safe_min(values):
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    return float(np.nanmin(values))


def safe_median(values):
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    return float(np.nanmedian(values))


def safe_percentile(values, percentile):
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    return float(np.nanpercentile(values, percentile))


def circularity_from_geom(geom):
    area = geom.area
    perimeter = geom.length
    if perimeter <= 0 or area <= 0:
        return 0.0
    return float(4.0 * np.pi * area / (perimeter * perimeter))


def rotated_rectangle_axes(geom):
    try:
        rect = geom.minimum_rotated_rectangle
        coords = list(rect.exterior.coords)
        if len(coords) < 5:
            return 0.0, 0.0

        lengths = []
        for i in range(4):
            x1, y1 = coords[i]
            x2, y2 = coords[i + 1]
            lengths.append(((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5)

        positive_lengths = [v for v in lengths if v > 0]
        if not positive_lengths:
            return 0.0, 0.0

        return float(max(positive_lengths)), float(min(positive_lengths))
    except Exception:
        return 0.0, 0.0


def elongation_from_minimum_rotated_rectangle(geom):
    long_axis, short_axis = rotated_rectangle_axes(geom)
    if short_axis <= 0:
        return 999.0
    return float(long_axis / short_axis)


def minor_axis_from_minimum_rotated_rectangle(geom):
    long_axis, short_axis = rotated_rectangle_axes(geom)
    return float(short_axis)


def solidity_from_geom(geom):
    try:
        hull_area = geom.convex_hull.area
        if hull_area <= 0:
            return 0.0
        return float(geom.area / hull_area)
    except Exception:
        return 0.0


def linear_score(value, low, high):
    if value is None or not np.isfinite(value) or high <= low:
        return 0.0
    return float(np.clip((value - low) / (high - low), 0.0, 1.0))


def inverse_linear_score(value, good, bad):
    if value is None or not np.isfinite(value) or bad <= good:
        return 0.0
    return float(np.clip(1.0 - (value - good) / (bad - good), 0.0, 1.0))


def stats_for_mask(array, mask):
    values = array[mask & np.isfinite(array)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    return float(np.nanmean(values)), float(np.nanmax(values)), float(np.nanmedian(values))


def read_label_file(master_profile):
    if not GEOPANDAS_AVAILABLE:
        print("geopandas is not available. Label file cannot be read.")
        return None

    if not label_gpkg.exists():
        print("Label GeoPackage not found.")
        return None

    try:
        if label_layer is None:
            gdf = gpd.read_file(label_gpkg)
        else:
            gdf = gpd.read_file(label_gpkg, layer=label_layer)
    except Exception:
        gdf = gpd.read_file(label_gpkg)

    if gdf.empty:
        print("Label GeoPackage is empty.")
        return None

    if gdf.crs != master_profile["crs"]:
        gdf = gdf.to_crs(master_profile["crs"])

    return gdf.reset_index(drop=True)


def rasterize_training_labels(label_gdf, master_profile):
    positive_mask = np.zeros(
        (master_profile["height"], master_profile["width"]),
        dtype=bool,
    )
    negative_mask = np.zeros_like(positive_mask)

    if label_gdf is None or label_gdf.empty:
        return positive_mask, negative_mask

    if label_column not in label_gdf.columns:
        print(f"Column {label_column} was not found. Treating all label polygons as positive.")
        positive_gdf = label_gdf.copy()
        negative_gdf = label_gdf.iloc[0:0].copy()
    else:
        positive_gdf = label_gdf[label_gdf[label_column].isin(positive_label_values)].copy()
        negative_gdf = label_gdf[~label_gdf[label_column].isin(positive_label_values)].copy()

    if len(positive_gdf) > 0:
        positive_mask = rasterize(
            [(geom, 1) for geom in positive_gdf.geometry if geom is not None and not geom.is_empty],
            out_shape=(master_profile["height"], master_profile["width"]),
            transform=master_profile["transform"],
            fill=0,
            dtype="uint8",
            all_touched=True,
        ).astype(bool)

    if len(negative_gdf) > 0:
        negative_mask = rasterize(
            [(geom, 1) for geom in negative_gdf.geometry if geom is not None and not geom.is_empty],
            out_shape=(master_profile["height"], master_profile["width"]),
            transform=master_profile["transform"],
            fill=0,
            dtype="uint8",
            all_touched=True,
        ).astype(bool)

    print(f"Positive label polygons: {len(positive_gdf)}")
    print(f"Negative label polygons: {len(negative_gdf)}")

    return positive_mask, negative_mask


def read_known_positive_mounds(master_profile):
    label_gdf = read_label_file(master_profile)
    if label_gdf is None:
        return None
    if label_column in label_gdf.columns:
        label_gdf = label_gdf[label_gdf[label_column].isin(positive_label_values)].copy()
    if label_gdf.empty:
        return None
    return label_gdf.reset_index(drop=True)


def object_feature_row(
    candidate_id,
    obj_slice,
    component,
    probability_smooth,
    broad_support_smooth,
    elevation_like,
    slope,
    roughness,
    feature_arrays,
    pixel_size,
    pixel_area_m2,
):
    pixel_count = int(component.sum())
    area_m2 = pixel_count * pixel_area_m2

    prob_values = probability_smooth[obj_slice][component]
    broad_values = broad_support_smooth[obj_slice][component]
    elev_values = elevation_like[obj_slice][component]
    slope_values = slope[obj_slice][component]
    rough_values = roughness[obj_slice][component]

    rows, cols = np.where(component)

    if rows.size > 0 and cols.size > 0:
        height_pixels = int(rows.max() - rows.min() + 1)
        width_pixels = int(cols.max() - cols.min() + 1)

        bbox_height_m = height_pixels * pixel_size
        bbox_width_m = width_pixels * pixel_size

        major_axis_raster_m = max(bbox_height_m, bbox_width_m)
        minor_axis_raster_m = max(min(bbox_height_m, bbox_width_m), pixel_size)

        elongation_raster = major_axis_raster_m / minor_axis_raster_m
        bbox_area_m2 = bbox_height_m * bbox_width_m
        bbox_fill_ratio = area_m2 / bbox_area_m2 if bbox_area_m2 > 0 else np.nan
    else:
        bbox_height_m = np.nan
        bbox_width_m = np.nan
        major_axis_raster_m = np.nan
        minor_axis_raster_m = np.nan
        elongation_raster = np.nan
        bbox_fill_ratio = np.nan

    boundary_pixels = binary_dilation(component) & ~component
    boundary_m = int(boundary_pixels.sum()) * pixel_size

    if boundary_m > 0 and area_m2 > 0:
        raster_roundness = float(4.0 * np.pi * area_m2 / (boundary_m * boundary_m))
    else:
        raster_roundness = np.nan

    dist_inside = distance_transform_edt(component) * pixel_size

    prob_sub = probability_smooth[obj_slice].copy()
    prob_sub[~component] = np.nan

    if np.isfinite(prob_sub).any():
        peak_index = np.nanargmax(prob_sub)
        peak_row, peak_col = np.unravel_index(peak_index, prob_sub.shape)
        peak_edge_distance_m = float(dist_inside[peak_row, peak_col])
    else:
        peak_edge_distance_m = 0.0

    object_relief_m = safe_max(elev_values) - safe_percentile(elev_values, 10)

    row = {
        "candidate_id": candidate_id,
        "pixel_count": pixel_count,
        "area_m2_raster": area_m2,

        "mean_probability": safe_mean(prob_values),
        "max_probability": safe_max(prob_values),
        "median_probability": safe_median(prob_values),
        "p90_probability": safe_percentile(prob_values, 90),

        "mean_broad_support": safe_mean(broad_values),
        "max_broad_support": safe_max(broad_values),
        "median_broad_support": safe_median(broad_values),
        "p90_broad_support": safe_percentile(broad_values, 90),

        "mean_elevation_like": safe_mean(elev_values),
        "max_elevation_like": safe_max(elev_values),
        "min_elevation_like": safe_min(elev_values),
        "object_relief_m": object_relief_m,

        "mean_slope": safe_mean(slope_values),
        "p90_slope": safe_percentile(slope_values, 90),
        "max_slope": safe_max(slope_values),

        "mean_roughness": safe_mean(rough_values),
        "p90_roughness": safe_percentile(rough_values, 90),
        "max_roughness": safe_max(rough_values),

        "bbox_width_m": bbox_width_m,
        "bbox_height_m": bbox_height_m,
        "major_axis_raster_m": major_axis_raster_m,
        "minor_axis_raster_m": minor_axis_raster_m,
        "elongation_raster": elongation_raster,
        "bbox_fill_ratio": bbox_fill_ratio,
        "raster_roundness": raster_roundness,
        "peak_edge_distance_m": peak_edge_distance_m,
    }

    for name, arr in feature_arrays.items():
        if name in ["slope", "roughness"]:
            continue

        values = arr[obj_slice][component]

        row[f"mean_{name}"] = safe_mean(values)
        row[f"max_{name}"] = safe_max(values)
        row[f"p90_{name}"] = safe_percentile(values, 90)

    return row


def pseudo_negative_score(row):
    elong_bad = linear_score(row.get("elongation_raster", np.nan), 2.5, 8.0)
    round_bad = inverse_linear_score(row.get("raster_roundness", np.nan), 0.08, 0.35)
    fill_bad = inverse_linear_score(row.get("bbox_fill_ratio", np.nan), 0.15, 0.55)
    large_bad = linear_score(row.get("area_m2_raster", np.nan), 60000, 250000)
    low_peak_centrality = inverse_linear_score(row.get("peak_edge_distance_m", np.nan), 3.0, 35.0)

    return float(
        0.30 * elong_bad
        + 0.25 * round_bad
        + 0.20 * fill_bad
        + 0.15 * large_bad
        + 0.10 * low_peak_centrality
    )


def rule_based_object_probability(row):
    prob_score = linear_score(row.get("max_probability", np.nan), 0.30, 0.95)
    mean_score = linear_score(row.get("mean_probability", np.nan), 0.10, 0.60)
    broad_score = linear_score(row.get("mean_broad_support", np.nan), 0.08, 0.55)
    relief_score = linear_score(row.get("object_relief_m", np.nan), 4.0, 35.0)
    round_score = linear_score(row.get("raster_roundness", np.nan), 0.08, 0.35)
    elong_score = inverse_linear_score(row.get("elongation_raster", np.nan), 1.0, 4.0)
    peak_score = linear_score(row.get("peak_edge_distance_m", np.nan), 4.0, 35.0)

    return float(
        0.24 * prob_score
        + 0.18 * mean_score
        + 0.18 * broad_score
        + 0.12 * relief_score
        + 0.10 * round_score
        + 0.10 * elong_score
        + 0.08 * peak_score
    )


# ============================================================
# 7. Load bathymetry
# ============================================================

print(f"{now()} Loading bathymetry")

with rasterio.open(bathymetry_path) as src:
    bathymetry = src.read(1).astype("float32")
    master_profile = src.profile.copy()
    master_profile.update(driver="GTiff")

    if src.nodata is not None:
        bathymetry[bathymetry == src.nodata] = np.nan

bathymetry[~np.isfinite(bathymetry)] = np.nan

valid_data_mask = np.isfinite(bathymetry)

pixel_size_x = abs(master_profile["transform"].a)
pixel_size_y = abs(master_profile["transform"].e)
pixel_size = float((pixel_size_x + pixel_size_y) / 2.0)
pixel_area_m2 = pixel_size_x * pixel_size_y

print(f"Raster size: {master_profile['width']} x {master_profile['height']}")
print(f"Pixel size: {pixel_size_x:.3f} x {pixel_size_y:.3f} m")
print(f"Bathymetry median: {float(np.nanmedian(bathymetry)):.3f}")
print(f"Bathymetry sign mode: {bathymetry_sign_mode}")
print(f"Watershed available: {SKIMAGE_AVAILABLE}")

if bathymetry_sign_mode == "negative_depths":
    elevation_like = bathymetry.copy()
elif bathymetry_sign_mode == "positive_depths":
    elevation_like = -bathymetry.copy()
else:
    raise ValueError("bathymetry_sign_mode must be negative_depths or positive_depths.")

elevation_like[~valid_data_mask] = np.nan

write_float_raster(
    feature_dir / "elevation_like_used_for_detection.tif",
    elevation_like,
    master_profile,
)

edge_buffer_pixels = max(1, int(round(edge_buffer_m / pixel_size)))
edge_distance = distance_transform_edt(valid_data_mask)
interior_mask = valid_data_mask & (edge_distance >= edge_buffer_pixels)

write_uint8_raster(
    output_dir / "valid_interior_mask.tif",
    interior_mask,
    master_profile,
)


# ============================================================
# 8. Create derived features
# ============================================================

print(f"{now()} Creating local height features")

feature_arrays = {}
calibration_rows = []

for window_m in local_height_windows_m:
    window_pixels = max(3, int(round(window_m / pixel_size)))

    local_mean = mean_filter_nan(elevation_like, window_pixels)
    local_height = elevation_like - local_mean
    local_height[~valid_data_mask] = np.nan

    name = f"local_height_{int(window_m)}m"
    feature_arrays[name] = local_height.astype("float32")

    write_float_raster(
        feature_dir / f"{name}.tif",
        local_height,
        master_profile,
    )

print(f"{now()} Loading or deriving roughness")

if roughness_path.exists():
    roughness = read_and_align_raster(roughness_path, master_profile, Resampling.bilinear)
    print("Using roughness file")
else:
    roughness_pixels = max(3, int(round(roughness_window_m / pixel_size)))
    roughness = std_filter_nan(elevation_like, roughness_pixels)
    print("Roughness file not found. Derived local standard deviation from bathymetry.")

roughness[~valid_data_mask] = np.nan
feature_arrays["roughness"] = roughness.astype("float32")

write_float_raster(
    feature_dir / "roughness_used.tif",
    roughness,
    master_profile,
)

print(f"{now()} Loading or deriving slope")

if slope_path.exists():
    slope = read_and_align_raster(slope_path, master_profile, Resampling.bilinear)
    print("Using slope file")
else:
    dzdy, dzdx = np.gradient(elevation_like.astype("float32"), pixel_size_y, pixel_size_x)
    slope = np.degrees(np.arctan(np.sqrt(dzdx * dzdx + dzdy * dzdy))).astype("float32")
    print("Slope file not found. Derived slope from elevation like bathymetry.")

slope[~valid_data_mask] = np.nan
feature_arrays["slope"] = slope.astype("float32")

write_float_raster(
    feature_dir / "slope_used.tif",
    slope,
    master_profile,
)


# ============================================================
# 9. Build fuzzy memberships and probability
# ============================================================

print(f"{now()} Building fuzzy memberships")

support_weighted = np.zeros(bathymetry.shape, dtype="float32")
support_weights = np.zeros(bathymetry.shape, dtype="float32")

broad_weighted = np.zeros(bathymetry.shape, dtype="float32")
broad_weights = np.zeros(bathymetry.shape, dtype="float32")

calibration_mask = interior_mask.copy()
calibration_notes = []

if use_calibration_mask:
    print(f"{now()} Creating calibration mask to ignore extreme tectonic ridges")

    rough_array = feature_arrays["roughness"]
    valid_rough = rough_array[interior_mask & np.isfinite(rough_array)]

    if valid_rough.size > 100:
        rough_cutoff = np.nanpercentile(valid_rough, calibration_ignore_roughest_percentile)
        calibration_mask &= np.isfinite(rough_array) & (rough_array <= rough_cutoff)
        calibration_notes.append(
            f"roughness <= p{calibration_ignore_roughest_percentile} ({rough_cutoff:.6g})"
        )
        print(f"Calibration ignores roughness above {rough_cutoff:.3f}")

    if calibration_also_ignore_extreme_local_height:
        broadest_name = f"local_height_{int(max(local_height_windows_m))}m"
        broadest_array = feature_arrays.get(broadest_name)

        if broadest_array is not None:
            valid_height = broadest_array[calibration_mask & np.isfinite(broadest_array)]

            if valid_height.size > 100:
                height_cutoff = np.nanpercentile(
                    valid_height,
                    calibration_ignore_local_height_percentile,
                )

                calibration_mask &= np.isfinite(broadest_array) & (broadest_array <= height_cutoff)

                calibration_notes.append(
                    f"{broadest_name} <= p{calibration_ignore_local_height_percentile} ({height_cutoff:.6g})"
                )

                print(f"Calibration ignores extreme {broadest_name} above {height_cutoff:.3f}")

    calibration_pixel_count = int(calibration_mask.sum())
    interior_pixel_count = int(interior_mask.sum())

    print(f"Calibration pixels used: {calibration_pixel_count} of {interior_pixel_count}")

    if calibration_pixel_count < 0.25 * interior_pixel_count:
        print("WARNING: Calibration mask became very small. Falling back to interior mask.")
        calibration_mask = interior_mask.copy()
        calibration_notes.append("fallback_to_interior_mask")

else:
    print(f"{now()} Calibration mask disabled. Using full interior mask for percentiles.")
    calibration_notes.append("full_interior_mask")

write_uint8_raster(
    output_dir / "calibration_mask_used_for_percentiles.tif",
    calibration_mask,
    master_profile,
)

for spec in feature_settings:
    name = spec["name"]
    array = feature_arrays.get(name)

    if array is None:
        print(f"Feature {name} is missing and will be skipped.")
        continue

    membership_type = spec["membership"]
    group = spec.get("group", "support")
    weight = float(spec.get("weight", 1.0))

    if membership_type == "high":
        p_low, p_high = spec["percentiles"]

        membership, low_value, high_value = percentile_membership_high(
            array,
            calibration_mask,
            p_low,
            p_high,
        )

    elif membership_type == "low":
        p_low, p_high = spec["percentiles"]

        membership, low_value, high_value = percentile_membership_low(
            array,
            calibration_mask,
            p_low,
            p_high,
        )

    elif membership_type == "trapezoid":
        a, b, c, d = spec["trapezoid"]
        membership = trapezoid_membership(array, a, b, c, d)
        low_value = a
        high_value = d

    else:
        raise ValueError(f"Unknown membership type: {membership_type}")

    membership[~valid_data_mask] = np.nan

    write_float_raster(
        membership_dir / f"membership_{name}.tif",
        membership,
        master_profile,
    )

    finite = np.isfinite(membership)

    support_weighted[finite] += membership[finite] * weight
    support_weights[finite] += weight

    if group == "broad":
        broad_weighted[finite] += membership[finite] * weight
        broad_weights[finite] += weight

    calibration_rows.append(
        {
            "feature": name,
            "membership": membership_type,
            "group": group,
            "weight": weight,
            "calibration_low": low_value,
            "calibration_high": high_value,
            "calibration_mask": "; ".join(calibration_notes),
        }
    )

probability = np.full(bathymetry.shape, np.nan, dtype="float32")
good = support_weights > 0
probability[good] = support_weighted[good] / support_weights[good]
probability[~valid_data_mask] = np.nan

broad_support = np.full(bathymetry.shape, np.nan, dtype="float32")
good_broad = broad_weights > 0
broad_support[good_broad] = broad_weighted[good_broad] / broad_weights[good_broad]
broad_support[~valid_data_mask] = np.nan

probability_smooth = gaussian_filter_nan(probability, smooth_probability_sigma_pixels)
broad_support_smooth = gaussian_filter_nan(broad_support, smooth_broad_support_sigma_pixels)

write_float_raster(
    output_dir / "obia_probability_raw.tif",
    probability,
    master_profile,
)

write_float_raster(
    output_dir / "obia_probability_smooth.tif",
    probability_smooth,
    master_profile,
)

write_float_raster(
    output_dir / "obia_broad_support_raw.tif",
    broad_support,
    master_profile,
)

write_float_raster(
    output_dir / "obia_broad_support_smooth_last.tif",
    broad_support_smooth,
    master_profile,
)

pd.DataFrame(calibration_rows).to_csv(
    output_dir / "feature_calibration.csv",
    index=False,
)


# ============================================================
# 10. Label assisted candidate extraction
# ============================================================

print(f"{now()} Starting label assisted candidate extraction")

label_gdf = read_label_file(master_profile)
positive_label_mask, negative_label_mask = rasterize_training_labels(label_gdf, master_profile)

write_uint8_raster(
    output_dir / "training_positive_label_mask.tif",
    positive_label_mask,
    master_profile,
)

write_uint8_raster(
    output_dir / "training_negative_label_mask.tif",
    negative_label_mask,
    master_profile,
)

candidate_mask = (
    valid_data_mask
    & interior_mask
    & np.isfinite(probability_smooth)
    & np.isfinite(broad_support_smooth)
    & (probability_smooth >= candidate_probability_threshold)
    & (broad_support_smooth >= candidate_broad_threshold)
)

# Labelled mound pixels are kept in the extraction mask so training positives are not lost.
if positive_label_mask.any():
    candidate_mask = candidate_mask | (positive_label_mask & valid_data_mask & interior_mask)

closing_radius_pixels = max(1, int(round(closing_radius_m / pixel_size)))
opening_radius_pixels = max(1, int(round(opening_radius_m / pixel_size)))

if closing_radius_m > 0:
    candidate_mask = binary_closing(
        candidate_mask,
        structure=disk_structure(closing_radius_pixels),
    )

if fill_holes_before_splitting:
    candidate_mask = binary_fill_holes(candidate_mask)

if opening_radius_m > 0:
    candidate_mask = binary_opening(
        candidate_mask,
        structure=disk_structure(opening_radius_pixels),
    )

candidate_mask = candidate_mask & valid_data_mask & interior_mask

write_uint8_raster(
    output_dir / "label_assisted_candidate_mask_before_split.tif",
    candidate_mask,
    master_profile,
)

# First create normal connected objects. Watershed is applied only where it is needed.
pre_labels, pre_object_count = connected_label(candidate_mask)
print(f"Connected objects before selective splitting: {pre_object_count}")

seed_peak_mask = np.zeros(candidate_mask.shape, dtype=bool)
labels = np.zeros(candidate_mask.shape, dtype="int32")
next_label_id = 1

if use_peak_splitting and SKIMAGE_AVAILABLE:
    print(f"{now()} Selective splitting of large or complex candidate objects")

    peak_distance_pixels = max(3, int(round(minimum_peak_distance_m / pixel_size)))
    if peak_distance_pixels % 2 == 0:
        peak_distance_pixels += 1

    max_radius_pixels = max(5, int(round(maximum_growth_radius_m / pixel_size)))

    split_surface = (
        0.70 * np.nan_to_num(probability_smooth, nan=0.0)
        + 0.30 * np.nan_to_num(broad_support_smooth, nan=0.0)
    ).astype("float32")

    split_surface = gaussian_filter_nan(split_surface, split_surface_sigma_pixels)
    split_surface[~candidate_mask] = np.nan

    write_float_raster(
        output_dir / "label_assisted_split_surface.tif",
        split_surface,
        master_profile,
    )

    pre_slices = find_objects(pre_labels)
    split_object_count = 0
    kept_unsplit_count = 0

    for original_id, obj_slice in enumerate(pre_slices, start=1):
        if obj_slice is None:
            continue

        component = pre_labels[obj_slice] == original_id
        pixel_count = int(component.sum())

        if pixel_count == 0:
            continue

        area_m2 = pixel_count * pixel_area_m2
        rows, cols = np.where(component)

        bbox_height_m = (rows.max() - rows.min() + 1) * pixel_size
        bbox_width_m = (cols.max() - cols.min() + 1) * pixel_size
        major_axis_m = max(bbox_height_m, bbox_width_m)
        minor_axis_m = max(min(bbox_height_m, bbox_width_m), pixel_size)
        elongation = major_axis_m / minor_axis_m

        should_try_split = (
            use_selective_peak_splitting
            and (
                area_m2 >= split_only_area_above_m2
                or major_axis_m >= split_only_major_axis_above_m
                or elongation >= split_only_elongation_above
            )
        )

        if not should_try_split:
            sub_out = labels[obj_slice]
            sub_out[component] = next_label_id
            labels[obj_slice] = sub_out
            next_label_id += 1
            kept_unsplit_count += 1
            continue

        sub_surface = split_surface[obj_slice].copy()
        sub_probability = probability_smooth[obj_slice]
        sub_broad = broad_support_smooth[obj_slice]

        local_maximum = maximum_filter(
            np.nan_to_num(sub_surface, nan=-9999.0),
            size=peak_distance_pixels,
            mode="nearest",
        )

        local_peak_mask = (
            component
            & np.isfinite(sub_surface)
            & (sub_surface == local_maximum)
            & np.isfinite(sub_probability)
            & np.isfinite(sub_broad)
            & (sub_probability >= seed_peak_probability_threshold)
            & (sub_broad >= seed_peak_broad_threshold)
        )

        if h_maxima is not None and seed_peak_prominence > 0:
            prominence_peaks = h_maxima(
                np.nan_to_num(sub_surface, nan=0.0).astype("float32"),
                seed_peak_prominence,
            ).astype(bool)

            local_peak_mask = local_peak_mask & prominence_peaks

        seed_sub_labels, seed_count = connected_label(local_peak_mask)

        if seed_count <= 1:
            sub_out = labels[obj_slice]
            sub_out[component] = next_label_id
            labels[obj_slice] = sub_out
            seed_peak_mask[obj_slice] |= local_peak_mask
            next_label_id += 1
            kept_unsplit_count += 1
            continue

        watershed_sub = watershed(
            image=-np.nan_to_num(sub_surface, nan=0.0),
            markers=seed_sub_labels,
            mask=component,
            watershed_line=False,
        ).astype("int32")

        distance_to_seed = distance_transform_edt(~local_peak_mask)
        watershed_sub[distance_to_seed > max_radius_pixels] = 0

        unique_sub_ids = [int(v) for v in np.unique(watershed_sub) if int(v) > 0]

        if len(unique_sub_ids) <= 1:
            sub_out = labels[obj_slice]
            sub_out[component] = next_label_id
            labels[obj_slice] = sub_out
            seed_peak_mask[obj_slice] |= local_peak_mask
            next_label_id += 1
            kept_unsplit_count += 1
            continue

        sub_out = labels[obj_slice]

        for sub_id in unique_sub_ids:
            piece = watershed_sub == sub_id

            if int(piece.sum()) * pixel_area_m2 < minimum_candidate_area_m2:
                continue

            sub_out[piece] = next_label_id
            next_label_id += 1

        labels[obj_slice] = sub_out
        seed_peak_mask[obj_slice] |= local_peak_mask
        split_object_count += 1

    print(f"Objects kept without watershed: {kept_unsplit_count}")
    print(f"Objects split with watershed: {split_object_count}")
    print(f"Seed peaks used after filtering: {int(seed_peak_mask.sum())}")

else:
    print(f"{now()} Using connected components without watershed")
    labels = pre_labels.astype("int32")

labels[~valid_data_mask] = 0
labels[~interior_mask] = 0

write_uint8_raster(
    output_dir / "label_assisted_seed_peak_mask.tif",
    seed_peak_mask,
    master_profile,
)

write_int32_raster(
    output_dir / "label_assisted_candidate_labels_initial.tif",
    labels,
    master_profile,
)

print(f"Initial candidate labels: {int(labels.max())}")


# ============================================================
# 11. Build object feature table
# ============================================================

print(f"{now()} Building object feature table")

slices = find_objects(labels)

filtered_labels = np.zeros(labels.shape, dtype="int32")
object_rows = []
candidate_id = 1

for original_id, obj_slice in enumerate(slices, start=1):
    if obj_slice is None:
        continue

    component = labels[obj_slice] == original_id

    if int(component.sum()) == 0:
        continue

    area_m2 = int(component.sum()) * pixel_area_m2

    if area_m2 < minimum_candidate_area_m2:
        continue

    if area_m2 > maximum_candidate_area_m2:
        continue

    row = object_feature_row(
        candidate_id,
        obj_slice,
        component,
        probability_smooth,
        broad_support_smooth,
        elevation_like,
        slope,
        roughness,
        feature_arrays,
        pixel_size,
        pixel_area_m2,
    )

    pos_overlap_pixels = int((positive_label_mask[obj_slice] & component).sum())
    neg_overlap_pixels = int((negative_label_mask[obj_slice] & component).sum())

    pos_overlap_area = pos_overlap_pixels * pixel_area_m2
    neg_overlap_area = neg_overlap_pixels * pixel_area_m2

    object_pixels = int(component.sum())

    pos_overlap_fraction = pos_overlap_pixels / object_pixels if object_pixels > 0 else 0.0
    neg_overlap_fraction = neg_overlap_pixels / object_pixels if object_pixels > 0 else 0.0

    row["positive_overlap_area_m2"] = pos_overlap_area
    row["negative_overlap_area_m2"] = neg_overlap_area
    row["positive_overlap_fraction"] = pos_overlap_fraction
    row["negative_overlap_fraction"] = neg_overlap_fraction
    row["pseudo_negative_score"] = pseudo_negative_score(row)

    if (
        pos_overlap_area >= min_positive_overlap_area_m2
        or pos_overlap_fraction >= min_positive_overlap_fraction
    ):
        row["training_label"] = 1
        row["training_label_source"] = "positive_polygon"

    elif (
        neg_overlap_area >= min_negative_overlap_area_m2
        or neg_overlap_fraction >= min_negative_overlap_fraction
    ):
        row["training_label"] = 0
        row["training_label_source"] = "negative_polygon"

    else:
        row["training_label"] = np.nan
        row["training_label_source"] = ""

    sub = filtered_labels[obj_slice]
    sub[component] = candidate_id
    filtered_labels[obj_slice] = sub

    object_rows.append(row)
    candidate_id += 1

object_df = pd.DataFrame(object_rows)

write_int32_raster(
    output_dir / "label_assisted_candidate_labels_before_ml.tif",
    filtered_labels,
    master_profile,
)

write_uint8_raster(
    output_dir / "label_assisted_candidate_mask_before_ml.tif",
    filtered_labels > 0,
    master_profile,
)

object_df.to_csv(
    output_dir / "label_assisted_candidate_object_features_before_ml.csv",
    index=False,
)

print(f"Candidate objects after size filtering: {len(object_df)}")

if len(object_df) == 0:
    raise RuntimeError("No candidate objects were extracted. Lower candidate_probability_threshold or candidate_broad_threshold.")


# ============================================================
# 12. Add pseudo negative examples if needed
# ============================================================

print(f"{now()} Preparing training labels")

training_counts = object_df["training_label"].value_counts(dropna=False)
print("Training labels before pseudo negatives:")
print(training_counts)

positive_count = int((object_df["training_label"] == 1).sum())
negative_count = int((object_df["training_label"] == 0).sum())

if create_pseudo_negatives and positive_count >= minimum_training_objects_per_class and negative_count < minimum_training_objects_per_class:
    print("Creating pseudo negative examples from likely false positive objects.")

    unlabeled_mask = (
        object_df["training_label"].isna()
        & (object_df["positive_overlap_area_m2"] == 0)
    )

    pseudo_pool = object_df[unlabeled_mask].copy()

    if len(pseudo_pool) > 0:
        needed = max(
            minimum_training_objects_per_class - negative_count,
            min(positive_count * 2, maximum_pseudo_negative_objects),
        )

        needed = min(needed, len(pseudo_pool), maximum_pseudo_negative_objects)

        pseudo_ids = (
            pseudo_pool
            .sort_values("pseudo_negative_score", ascending=False)
            .head(needed)["candidate_id"]
            .astype(int)
            .tolist()
        )

        object_df.loc[
            object_df["candidate_id"].isin(pseudo_ids),
            "training_label",
        ] = 0

        object_df.loc[
            object_df["candidate_id"].isin(pseudo_ids),
            "training_label_source",
        ] = "pseudo_negative"

training_counts = object_df["training_label"].value_counts(dropna=False)
print("Training labels after pseudo negatives:")
print(training_counts)


# ============================================================
# 13. Train object classifier or use fallback score
# ============================================================

feature_columns = [
    col for col in object_df.columns
    if col not in [
        "candidate_id",
        "training_label",
        "training_label_source",
        "positive_overlap_area_m2",
        "negative_overlap_area_m2",
        "positive_overlap_fraction",
        "negative_overlap_fraction",
    ]
]

feature_df = object_df[feature_columns].copy()
feature_df = feature_df.replace([np.inf, -np.inf], np.nan)

feature_medians = feature_df.median(numeric_only=True)
feature_df = feature_df.fillna(feature_medians)
feature_df = feature_df.fillna(0)

training_mask = object_df["training_label"].isin([0, 1])
X_train_all = feature_df.loc[training_mask, feature_columns]
y_train_all = object_df.loc[training_mask, "training_label"].astype(int)

class_counts = y_train_all.value_counts()

can_train_classifier = (
    len(class_counts) == 2
    and class_counts.min() >= minimum_training_objects_per_class
)

if can_train_classifier:
    print(f"{now()} Training Random Forest object classifier")
    print(class_counts)

    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=random_seed,
        class_weight="balanced",
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
    )

    if len(y_train_all) >= 12 and class_counts.min() >= 4:
        X_train, X_test, y_train, y_test = train_test_split(
            X_train_all,
            y_train_all,
            test_size=0.30,
            random_state=random_seed,
            stratify=y_train_all,
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)
        y_prob = rf.predict_proba(X_test)[:, 1]

        report = classification_report(
            y_test,
            y_pred,
            zero_division=0,
        )

        print(report)
        print("Confusion matrix:")
        print(confusion_matrix(y_test, y_pred))

        try:
            auc = roc_auc_score(y_test, y_prob)
            print(f"Validation ROC AUC: {auc:.3f}")
        except Exception:
            auc = np.nan

        with open(output_dir / "object_classifier_validation_report.txt", "w") as f:
            f.write(report)
            f.write("\nConfusion matrix:\n")
            f.write(str(confusion_matrix(y_test, y_pred)))
            f.write(f"\nROC AUC: {auc}\n")

    rf.fit(X_train_all, y_train_all)

    object_df["ml_mound_probability"] = rf.predict_proba(feature_df[feature_columns])[:, 1]

    importance_df = pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": rf.feature_importances_,
        }
    ).sort_values("importance", ascending=False)

    importance_df.to_csv(
        output_dir / "object_classifier_feature_importance.csv",
        index=False,
    )

    joblib.dump(
        {
            "model": rf,
            "feature_columns": feature_columns,
            "feature_medians": feature_medians,
            "threshold": final_object_probability_threshold,
        },
        object_model_path,
    )

else:
    print("Not enough labelled positive and negative objects to train a classifier.")
    print("Using fallback object probability score.")

    object_df["ml_mound_probability"] = object_df.apply(
        rule_based_object_probability,
        axis=1,
    )

    pd.DataFrame(
        {
            "feature": ["fallback_rule_based_score"],
            "importance": [1.0],
        }
    ).to_csv(
        output_dir / "object_classifier_feature_importance.csv",
        index=False,
    )

object_df["ml_prediction"] = (
    object_df["ml_mound_probability"] >= final_object_probability_threshold
).astype(int)

if force_keep_training_positives:
    object_df.loc[object_df["training_label"] == 1, "ml_prediction"] = 1
    object_df.loc[object_df["training_label"] == 1, "ml_mound_probability"] = np.maximum(
        object_df.loc[object_df["training_label"] == 1, "ml_mound_probability"],
        final_object_probability_threshold,
    )

object_df.to_csv(
    output_dir / "label_assisted_candidate_object_features_after_ml.csv",
    index=False,
)


# ============================================================
# 14. Create final raster labels
# ============================================================

print(f"{now()} Creating final candidate raster")

keep_ids = set(
    object_df.loc[
        object_df["ml_prediction"] == 1,
        "candidate_id",
    ].astype(int).tolist()
)

final_labels = np.zeros(filtered_labels.shape, dtype="int32")
new_final_id = 1
final_lookup_rows = []

for old_id in sorted(keep_ids):
    mask_old = filtered_labels == old_id

    if int(mask_old.sum()) == 0:
        continue

    final_labels[mask_old] = new_final_id

    row = object_df[object_df["candidate_id"] == old_id].iloc[0].to_dict()
    row["final_id"] = new_final_id
    final_lookup_rows.append(row)

    new_final_id += 1

final_df = pd.DataFrame(final_lookup_rows)

final_df.to_csv(
    output_dir / "label_assisted_final_candidate_object_features.csv",
    index=False,
)

write_int32_raster(
    output_dir / "label_assisted_candidate_labels_final.tif",
    final_labels,
    master_profile,
)

write_uint8_raster(
    output_dir / "label_assisted_candidate_mask_final_try.tif",
    final_labels > 0,
    master_profile,
)

print(f"Final candidate objects: {len(final_df)}")


# ============================================================
# 15. Vectorise candidates
# ============================================================

if GEOPANDAS_AVAILABLE:
    print(f"{now()} Vectorising candidates")

    all_records = []

    for geom_mapping, value in shapes(
        filtered_labels,
        mask=filtered_labels > 0,
        transform=master_profile["transform"],
    ):
        candidate_id_value = int(value)
        geom = shape(geom_mapping)

        if polygon_simplify_tolerance_m > 0:
            geom = geom.simplify(
                polygon_simplify_tolerance_m,
                preserve_topology=True,
            )

        if geom.is_empty:
            continue

        all_records.append(
            {
                "candidate_id": candidate_id_value,
                "geometry": geom,
            }
        )

    if all_records:
        all_gdf = gpd.GeoDataFrame(
            all_records,
            crs=master_profile["crs"],
        )

        all_gdf = all_gdf.dissolve(
            by="candidate_id",
            as_index=False,
        )

        all_gdf = all_gdf.merge(
            object_df,
            on="candidate_id",
            how="left",
        )

        all_gdf["area_m2_vector"] = all_gdf.geometry.area
        all_gdf["roundness_vector"] = all_gdf.geometry.apply(circularity_from_geom)
        all_gdf["elongation_vector"] = all_gdf.geometry.apply(elongation_from_minimum_rotated_rectangle)
        all_gdf["minor_axis_m_vector"] = all_gdf.geometry.apply(minor_axis_from_minimum_rotated_rectangle)
        all_gdf["solidity_vector"] = all_gdf.geometry.apply(solidity_from_geom)

        all_gdf.to_file(
            output_dir / "label_assisted_candidates_all_before_ml_filter.gpkg",
            layer="candidates_all",
            driver="GPKG",
        )

        all_gdf.drop(columns="geometry").to_csv(
            output_dir / "label_assisted_candidates_all_before_ml_filter.csv",
            index=False,
        )

        final_gdf = all_gdf[
            all_gdf["ml_prediction"] == 1
        ].copy()

        final_gdf = final_gdf.sort_values(
            "ml_mound_probability",
            ascending=False,
        ).reset_index(drop=True)

        final_gdf["final_id"] = np.arange(1, len(final_gdf) + 1)

        if len(final_gdf) > 0:
            final_gdf.to_file(
                output_dir / "label_assisted_candidates_final.gpkg",
                layer="candidates_final",
                driver="GPKG",
            )

        final_gdf.drop(columns="geometry").to_csv(
            output_dir / "label_assisted_candidates_final.csv",
            index=False,
        )

else:
    print("geopandas unavailable. Skipping vector outputs.")


# ============================================================
# 16. Known mound diagnostics
# ============================================================

if run_known_mound_diagnostics:
    print(f"{now()} Running known mound diagnostics")

    known_gdf = read_known_positive_mounds(master_profile)

    if known_gdf is not None and len(known_gdf) > 0:
        diagnostic_rows = []
        final_mask = final_labels > 0

        name_candidates = [
            "name",
            "Name",
            "mound",
            "Mound",
            "site",
            "Site",
            "target",
            "Target",
        ]

        for idx, row in known_gdf.iterrows():
            geom = row.geometry

            mound_mask = geometry_mask(
                [geom],
                out_shape=(master_profile["height"], master_profile["width"]),
                transform=master_profile["transform"],
                invert=True,
                all_touched=True,
            )

            mound_mask = mound_mask & valid_data_mask

            mean_prob, max_prob, median_prob = stats_for_mask(
                probability_smooth,
                mound_mask,
            )

            mean_broad, max_broad, median_broad = stats_for_mask(
                broad_support_smooth,
                mound_mask,
            )

            mound_pixels = int(mound_mask.sum())
            overlap_pixels = int((mound_mask & final_mask).sum())

            overlap_percent = np.nan
            recovered = False

            if mound_pixels > 0:
                overlap_percent = 100.0 * overlap_pixels / mound_pixels
                recovered = overlap_pixels > 0

            mound_name = f"known_{idx + 1}"

            for col in name_candidates:
                if col in known_gdf.columns and pd.notna(row[col]):
                    mound_name = str(row[col])
                    break

            diagnostic_rows.append(
                {
                    "known_id": idx + 1,
                    "known_name": mound_name,
                    "area_m2": geom.area,
                    "mean_probability": mean_prob,
                    "max_probability": max_prob,
                    "median_probability": median_prob,
                    "mean_broad_support": mean_broad,
                    "max_broad_support": max_broad,
                    "median_broad_support": median_broad,
                    "overlap_percent_final_candidates": overlap_percent,
                    "recovered_by_final_candidates": recovered,
                }
            )

        diagnostics_df = pd.DataFrame(diagnostic_rows)

        diagnostics_df.to_csv(
            output_dir / "known_mound_diagnostics_label_assisted.csv",
            index=False,
        )

        print("Known mound diagnostics written to known_mound_diagnostics_label_assisted.csv")
        print(
            diagnostics_df[
                [
                    "known_name",
                    "max_probability",
                    "max_broad_support",
                    "overlap_percent_final_candidates",
                    "recovered_by_final_candidates",
                ]
            ].to_string(index=False)
        )

    else:
        print("Known mound diagnostics skipped because no positive label polygons were available.")


# ============================================================
# 17. Final summary
# ============================================================

print("")
print("Finished.")
print(f"Outputs written to: {output_dir}")
print("")
print("Load these in QGIS first:")
print("1. obia_probability_smooth.tif")
print("2. obia_broad_support_smooth.tif")
print("3. label_assisted_candidate_mask_before_split.tif")
print("4. label_assisted_candidate_mask_before_ml.tif")
print("5. label_assisted_candidate_mask_final.tif")
print("6. label_assisted_candidates_all_before_ml_filter.gpkg")
print("7. label_assisted_candidates_final.gpkg")
print("8. label_assisted_candidate_object_features_after_ml.csv")
print("9. object_classifier_feature_importance.csv")
print("10. known_mound_diagnostics_label_assisted.csv")

14:25:33 Loading bathymetry
Raster size: 4985 x 5067
Pixel size: 1.888 x 1.888 m
Bathymetry median: -3458.500
Bathymetry sign mode: negative_depths
Watershed available: True
14:25:50 Creating local height features
14:27:05 Loading or deriving roughness
Using roughness file
14:27:15 Loading or deriving slope
Using slope file
14:27:24 Building fuzzy memberships
14:27:24 Creating calibration mask to ignore extreme tectonic ridges
Calibration ignores roughness above 3.023
Calibration ignores extreme local_height_400m above 24.295
Calibration pixels used: 10177025 of 12342902
14:29:14 Starting label assisted candidate extraction
Positive label polygons: 59
Negative label polygons: 42
Connected objects before selective splitting: 896
14:29:21 Selective splitting of large or complex candidate objects
Objects kept without watershed: 673
Objects split with watershed: 223
Seed peaks used after filtering: 2796
Initial candidate labels: 3349
14:30:17 Building object feature table
Candidate objects